# Gold — Geography Dimension
Single-row geography dimension for Iceland. Included per Kimball conventions.

**Output:** `gold.dim_geography` (Delta table)  
**Grain:** One row per country

In [ ]:
import pandas as pd

geographies = [
    (1, "IS", "Iceland", "Northern Europe", "ISK", "Icelandic Króna"),
]

df_geo = pd.DataFrame(geographies, columns=[
    "geography_key", "country_iso_code", "country_name",
    "region", "currency_code", "currency_name"
])

In [ ]:
df_spark = spark.createDataFrame(df_geo)
df_spark.createOrReplaceTempView("v_dim_geography")

if not spark.catalog.tableExists("gold.dim_geography"):
    df_spark.write.format("delta").saveAsTable("gold.dim_geography")
else:
    spark.sql("""
        MERGE INTO gold.dim_geography AS target
        USING v_dim_geography AS source
        ON target.geography_key = source.geography_key
        WHEN MATCHED THEN UPDATE SET
            target.country_iso_code = source.country_iso_code,
            target.country_name     = source.country_name,
            target.region           = source.region,
            target.currency_code    = source.currency_code,
            target.currency_name    = source.currency_name
        WHEN NOT MATCHED THEN INSERT (
            geography_key, country_iso_code, country_name,
            region, currency_code, currency_name
        )
        VALUES (
            source.geography_key, source.country_iso_code, source.country_name,
            source.region, source.currency_code, source.currency_name
        )
    """)